In [2]:
!pip install rouge-score bert-score json-repair bitsandbytes>=0.46.1 evaluate

In [3]:
from google.colab import userdata
from datasets import load_dataset, DatasetDict, Dataset
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForCausalLM
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import torch
import json
import random
import json_repair
import pandas as pd
import numpy as np
from google.colab import drive
import evaluate
from collections import defaultdict
import os
import gc
drive.mount('/content/drive')
device = 'cuda'

Mounted at /content/drive


In [4]:
folder_path = '/content/drive/MyDrive/facetsum/dataset'

train_df = pd.read_parquet(f'{folder_path}/train.parquet')
val_df = pd.read_parquet(f'{folder_path}/val.parquet')
test_df = pd.read_parquet(f'{folder_path}/test.parquet')

In [5]:
train_df = Dataset.from_pandas(train_df)
val_df = Dataset.from_pandas(val_df)
test_df = Dataset.from_pandas(test_df)

df = DatasetDict({
    "train": train_df,
    "validation": val_df,
    "test": test_df
})

df

DatasetDict({
    train: Dataset({
        features: ['title', 'keywords', 'url', 'section_names', 'sections', 'abstract_sections_names', 'abstract_sections', 'references', 'appendix', 'journal', 'id', 'category', 'paper_word_count', '__index_level_0__'],
        num_rows: 22086
    })
    validation: Dataset({
        features: ['title', 'keywords', 'url', 'section_names', 'sections', 'abstract_sections_names', 'abstract_sections', 'references', 'appendix', 'journal', 'id', 'category', 'paper_word_count', '__index_level_0__'],
        num_rows: 2761
    })
    test: Dataset({
        features: ['title', 'keywords', 'url', 'section_names', 'sections', 'abstract_sections_names', 'abstract_sections', 'references', 'appendix', 'journal', 'id', 'category', 'paper_word_count', '__index_level_0__'],
        num_rows: 2761
    })
})

In [6]:
class ModelAnswer(BaseModel):
    Purpose: str = Field(
        ...,
        description="\n".join([
                "Summarize the main purpose of the research paper.",
                "Explain why this study was conducted, what problem it aims to solve.",
                "Focus on the research goal and background, not the methodology or results."
            ])
        )

    Method: str = Field(
        ...,
        description="\n".join([
                "Summarize the methodology used in the paper.",
                "Include the approach, experimental design, models, tools, datasets, protocols, and techniques used to achieve the research objective.",
                "Focus on how the problem was solved."
            ])
    )

    Findings: str = Field(
        ...,
        description="\n".join([
                "Summarize the main findings and results of the research.",
                "Include key outcomes, experimental results, answers to research questions, hypothesis validation, performance improvements, and important observations.",
                "Focus on what the researchers discovered."
            ])
    )

    Value: str = Field(
        ...,
        description="\n".join([
                "Summarize the value and contribution of the research paper.",
                "Explain what is novel, original, or important about this work.",
                "How it contributes to the field, and why it matters compared to previous work.",
                "Focus on the significance and originality of the study."
            ])
    )

In [7]:
system_prompt = "\n".join([
    "You are an expert AI assistant for the 'Mirath' platform, specializing in academic paper summarization.",
    "Your task is to analyze full research papers and extract a structured summary into valid JSON.",
    "Strictly follow the provided Pydantic schema for the JSON structure.",
    "Output ONLY the raw JSON string. Do not include markdown blocks, explanations, or text outside the JSON.",
    "Maintain a formal, objective academic tone.",
    "Ensure distinct information for each field (Purpose, Method, Findings, Value) with no redundancy."
])

In [8]:
def parse_json(text):
    try:
        return json_repair.loads(text)
    except:
        return None

In [ ]:

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
qwen = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
    quantization_config=bnb_config
)

In [12]:
def get_bins(dataset, step=1000):
  lengths = dataset["paper_word_count"]

  bins = {}
  for i, l in enumerate(lengths):
    bucket = (l // step) * step
    bins.setdefault(bucket, []).append(i)

  return bins

In [13]:
def sample_from_bins(dataset, bins, n=3):
  samples = []

  for b, idxs in bins.items():
    chosen = random.sample(idxs, min(n, len(idxs)))

    for i in chosen:
      samples.append((b, dataset[i]))

  return samples

In [16]:
bins = get_bins(df["validation"], step=1000)
for k , v in bins.items():
  print(k)
  print(v)

5000
[0, 3, 8, 14, 16, 19, 28, 30, 31, 41, 44, 48, 51, 52, 53, 57, 58, 61, 67, 72, 73, 75, 76, 79, 98, 106, 115, 116, 117, 119, 124, 127, 131, 141, 156, 160, 161, 162, 167, 172, 177, 182, 191, 206, 208, 214, 228, 230, 232, 234, 235, 241, 243, 247, 249, 255, 263, 270, 271, 274, 278, 282, 283, 284, 289, 291, 312, 324, 329, 333, 336, 338, 340, 349, 355, 357, 370, 373, 374, 375, 383, 385, 390, 391, 399, 407, 408, 410, 416, 418, 422, 431, 436, 452, 453, 455, 460, 461, 466, 467, 469, 473, 474, 495, 496, 500, 503, 524, 527, 530, 533, 534, 539, 542, 543, 545, 552, 567, 570, 573, 578, 579, 590, 615, 633, 636, 641, 646, 648, 656, 672, 677, 692, 693, 698, 700, 710, 712, 713, 714, 723, 729, 734, 738, 739, 742, 758, 760, 763, 767, 769, 775, 776, 780, 784, 799, 801, 802, 804, 816, 817, 818, 826, 828, 829, 831, 837, 843, 844, 845, 848, 849, 853, 862, 863, 868, 869, 875, 878, 879, 883, 890, 891, 896, 898, 900, 911, 918, 920, 928, 929, 935, 936, 942, 946, 947, 948, 950, 952, 955, 956, 963, 967, 968, 97

In [22]:
samples = sample_from_bins(df["validation"], bins, n=1)
samples[0]

(5000,
 {'title': 'The impact of a design management training initiative on project performance',
  'keywords': ['Construction works',
   'Design management',
   'Procurement',
   'Training methods',
   'Project teams',
   'Project management'],
  'url': 'https://www.emerald.com/insight/content/doi/10.1108/09699980610646476/full/html',
  'section_names': ['Introduction',
   'Research methodology',
   'Design management awareness training',
   'Design management handbook, practices and tools',
   'Barriers to implementing practices and tools',
   'Design management handbook modifications',
   'Design management maturity assessment',
   'Conclusions'],
  'sections': [["Clients are increasingly adopting design and build type procurement routes in favour of traditional contracts to reduce their risks associated with construction projects. As a result, contractors are now expected to accept an increasing responsibility for the control of design - a process they have had little experience in

In [10]:
def get_full_paper_text(paper):
  paper_text = []
  for sec in (paper):
    for text in sec:
      paper_text.append(text)

  paper_text = "\n".join(paper_text)
  return paper_text

In [35]:
def generate_output(input):

  messages = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": "\n".join([
                  "## full paper text:",
                   get_full_paper_text(input),
                  "",

                  "## Pydantic Details:",
                  json.dumps(
                      ModelAnswer.model_json_schema(), ensure_ascii=False
                  ),
                  "",

                  "# Output JSON:",
                  "```json"
              ])
    },
    ]

  inputs = tokenizer.apply_chat_template(
  messages,
  add_generation_prompt=True,
  tokenize=True,
  return_dict=True,
  return_tensors="pt",
  ).to(device)


  print(f"\nInput tokens: {inputs['input_ids'].shape[-1]}")

  with torch.inference_mode():
    outputs = qwen.generate(**inputs, max_new_tokens=512)

  answer = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

  del inputs
  del outputs

  torch.cuda.empty_cache()
  gc.collect()

  return parse_json(answer)


In [36]:
def test_oom(samples):

  results = []

  for length_bin, sample in samples:

    torch.cuda.reset_peak_memory_stats()

    try:
      print(f"\nTesting bin: {length_bin}")

      output = generate_output(sample['sections'])

      peak = torch.cuda.max_memory_allocated() / 1024**3

      results.append({
          "bin": length_bin,
          "status": "ok",
          "peak_gb": peak
      })

    except RuntimeError as e:
      if "out of memory" in str(e).lower():

        results.append({
            "bin": length_bin,
            "status": "oom"
        })

      else:
        print(e)

    torch.cuda.empty_cache()
    gc.collect()

  return results


results = test_oom(samples)

results


Testing bin: 5000

Input tokens: 7085

Testing bin: 4000

Input tokens: 6067

Testing bin: 3000

Input tokens: 4793

Testing bin: 6000

Input tokens: 8836

Testing bin: 7000

Input tokens: 9211

Testing bin: 8000

Input tokens: 12394

Testing bin: 2000

Input tokens: 3987

Testing bin: 9000

Input tokens: 13054


[{'bin': 5000, 'status': 'ok', 'peak_gb': 6.980719566345215},
 {'bin': 4000, 'status': 'ok', 'peak_gb': 5.519737720489502},
 {'bin': 3000, 'status': 'ok', 'peak_gb': 3.999385356903076},
 {'bin': 6000, 'status': 'ok', 'peak_gb': 10.002717971801758},
 {'bin': 7000, 'status': 'ok', 'peak_gb': 10.728926181793213},
 {'bin': 8000, 'status': 'oom'},
 {'bin': 2000, 'status': 'ok', 'peak_gb': 3.2120108604431152},
 {'bin': 9000, 'status': 'oom'}]

# Evaluation using rougeL score




In [4]:
base_dir = "/content/drive/MyDrive/facetsum/dataset"
os.makedirs(base_dir, exist_ok=True)

valid_data_path = os.path.join(base_dir, "valid_evaluation.jsonl")
oom_data_path = os.path.join(base_dir, "oom_evaluation.jsonl")

In [41]:
def save_evaluation_set_answers(eval_set):

  abstract_sections = ['Purpose' , 'Method' , 'Findings' , 'Value']

  for rec in eval_set :

    answers = {}
    try :
      model_answer = generate_output(rec['sections'])
      answers['model_answer'] = model_answer

      refrence_answer = {}
      for idx, sec in enumerate(rec['abstract_sections']):
        refrence_answer[abstract_sections[idx]] = ' '.join(sec)

      answers['refrence_answer'] = refrence_answer

      with open(valid_data_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(answers, ensure_ascii=False) + "\n")

    except Exception as e:
      with open(oom_data_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(answers, ensure_ascii=False) + "\n")



In [ ]:
save_evaluation_set_answers(df['validation'].select(range(0, 25)))


Input tokens: 8268

Input tokens: 7420

Input tokens: 6514

Input tokens: 6948

Input tokens: 5699

Input tokens: 9439

Input tokens: 5811

Input tokens: 5141

Input tokens: 8698

Input tokens: 6587

Input tokens: 6347

Input tokens: 9132

Input tokens: 5889

Input tokens: 5125

Input tokens: 8109

Input tokens: 10697

Input tokens: 7449

Input tokens: 12463

Input tokens: 5279

Input tokens: 7565

Input tokens: 3985

Input tokens: 6815

Input tokens: 4742

Input tokens: 9946

Input tokens: 9523


In [14]:
rouge = evaluate.load("rouge")
sections = ["Purpose", "Method", "Findings", "Value"]

In [10]:
def compute_rouge_per_section():
  scores = {s: defaultdict(list) for s in sections}

  with open(valid_data_path, "r", encoding="utf-8") as f:
    for line in f:
      sample = json.loads(line)

      ref = sample["refrence_answer"]
      pred = sample["model_answer"]

      if isinstance(pred, list):
        pred = pred[0]

      for sec in sections:

        pred_text = pred.get(sec, "")

        if sec == "Value":
          ref_text = ref.get('Values', "")
        else:
          ref_text = ref.get(sec, "")


        result = rouge.compute(
            predictions=[pred_text],
            references=[ref_text],
            use_stemmer=True
        )

        scores[sec]["rougeL"].append(result["rougeL"])

  final_scores = {}
  for sec in sections:
    final_scores[sec] = {
        "rougeL_mean": sum(scores[sec]["rougeL"]) / len(scores[sec]["rougeL"])
    }

  return final_scores

In [11]:
final_scores = compute_rouge_per_section()
final_scores

{'Purpose': {'rougeL_mean': np.float64(0.27949289548039047)},
 'Method': {'rougeL_mean': np.float64(0.18904658084399525)},
 'Findings': {'rougeL_mean': np.float64(0.20451200096317)},
 'Value': {'rougeL_mean': np.float64(0.2068148574270199)}}

# Evaluation using Bert score

In [12]:

def compute_bertscore_per_section():
  scores = {s: defaultdict(list) for s in sections}

  with open(valid_data_path, "r", encoding="utf-8") as f:
    for line in f:
      sample = json.loads(line)

      ref = sample["refrence_answer"]
      pred = sample["model_answer"]

      if isinstance(pred, list):
        pred = pred[0]

      for sec in sections:

        pred_text = pred.get(sec, "")

        if sec == "Value":
          ref_text = ref.get("Values", "")
        else:
          ref_text = ref.get(sec, "")

        P, R, F1 = bert_score(
            [pred_text],
            [ref_text],
            lang="en",
            model_type="distilbert-base-uncased"
        )

        scores[sec]["bertscore_f1"].append(F1.item())

  final_scores = {}

  for sec in sections:
    final_scores[sec] = {
        "bertscore_f1_mean":
            sum(scores[sec]["bertscore_f1"]) /
            len(scores[sec]["bertscore_f1"])
    }

  return final_scores

In [ ]:
results = compute_bertscore_per_section()

In [14]:
results

{'Purpose': {'bertscore_f1_mean': 0.8255964064598084},
 'Method': {'bertscore_f1_mean': 0.795990583896637},
 'Findings': {'bertscore_f1_mean': 0.8056027269363404},
 'Value': {'bertscore_f1_mean': 0.8081868934631348}}

# Preparing the data for fine-tuning

In [ ]:
def prepare_data(data):

  llm_finetunning_data = []
  for rec in data:
    llm_finetunning_data.append({
        "system": system_prompt,
        "instruction":  "\n".join([
                        "## full paper text:",
                         get_full_paper_text(rec['sections']),
                        "",

                        "## Pydantic Details:",
                        json.dumps(
                            ModelAnswer.model_json_schema(), ensure_ascii=False
                        ),
                        "",

                        "# Output JSON:",
                        "```json"
              ]),
        "input": "",
        "output": "\n".join([
            "```json",
            f"Purpose: {' '.join(rec['abstract_sections'][0])}",
            f"Method: {' '.join(rec['abstract_sections'][1])}",
            f"Findings: {' '.join(rec['abstract_sections'][2])}",
            f"Value: {' '.join(rec['abstract_sections'][3])}",
            "```"
        ]),
        "history": []
    }
  )
  return llm_finetunning_data

In [ ]:
base_dir = "/content/drive/MyDrive/facetsum/finetunning_data"
os.makedirs(base_dir, exist_ok=True)

for obj in df:

  llm_finetunning_data = prepare_data(df[obj])
  data_path = os.path.join(base_dir, f"{obj}.json")

  with open(data_path, "w") as f:
    json.dump(llm_finetunning_data, f, indent=4, ensure_ascii=False)